# Classifieurs quantiques variationnels (VQC)
### Anatomie, entraînement, limites — et face-à-face avec les kernels quantiques

Ce notebook étudie en profondeur **une seule brique du QML** : les *circuits quantiques paramétrés* utilisés comme modèles d'apprentissage supervisé (on parle aussi de VQC, de PQC ou de *quantum neural networks*).

L'objectif n'est pas d'empiler des démos mais de comprendre **pourquoi** ces modèles se comportent comme ils le font. Chaque partie suit le même schéma : un peu de théorie → une expérience numérique → l'interprétation de ce qu'on observe.

| # | Partie | Question centrale |
|---|---|---|
| 1 | Anatomie d'un VQC | Que calcule exactement le modèle ? |
| 2 | Encodage et séries de Fourier | Quelles fonctions un VQC peut-il représenter ? |
| 3 | Gradients et *parameter-shift rule* | Comment dériver un circuit qu'on ne peut pas « dérouler » ? |
| 4 | Entraînement sur un vrai dataset | Comment l'encodage et la profondeur jouent-ils en pratique ? |
| 5 | *Barren plateaus* | Pourquoi l'entraînement devient-il impossible à grande échelle ? |
| 6 | Face-à-face | VQC vs kernel quantique vs modèles classiques |
| 7 | Synthèse | Ce qu'il faut retenir, exercices, références |

> **Lien avec les kernels.** La partie 1 montre qu'un VQC est un modèle *linéaire* dans l'espace des états quantiques, exactement comme un kernel quantique. La partie 6 met les deux approches côte à côte sur le même problème : ce notebook est conçu pour se lire en miroir d'un travail sur les kernels.

**Environnement** : simulations exactes avec PennyLane (`default.qubit`), NumPy, scikit-learn, pandas et matplotlib. L'exécution complète prend quelques minutes sur un portable.

In [ ]:
# Si nécessaire : %pip install pennylane scikit-learn matplotlib pandas
import time
import warnings
from functools import reduce

import numpy as onp
import pennylane as qml
from pennylane import numpy as pnp
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

warnings.filterwarnings("ignore")
SEED = 42
onp.random.seed(SEED)

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})
pd.set_option("display.precision", 4)

: 

---
## 1. Anatomie d'un VQC

Un VQC est une fonction $f_\theta : \mathbb{R}^d \to \mathbb{R}$ construite en trois étapes :

$$
f_\theta(x) \;=\; \langle 0 |\, S(x)^\dagger\, W(\theta)^\dagger \; O \; W(\theta)\, S(x) \,| 0 \rangle
$$

- **$S(x)$ — l'encodage** (*feature map*) : un circuit dont les angles dépendent des données, sans paramètre entraînable.
- **$W(\theta)$ — l'ansatz** : le circuit paramétré que l'on entraîne.
- **$O$ — l'observable** mesurée en sortie, ici $Z$ sur le premier qubit, donc $f_\theta(x) \in [-1, 1]$.

On ajoute ensuite un post-traitement classique : un biais, puis le signe pour classer.

### Le point de vue « matrice densité »

En notant $\rho(x) = S(x)|0\rangle\langle 0|S(x)^\dagger$ l'état encodé et $M_\theta = W(\theta)^\dagger O\, W(\theta)$ l'observable « tournée » par l'ansatz :

$$
f_\theta(x) = \operatorname{Tr}\big[\rho(x)\, M_\theta\big]
$$

C'est un **modèle linéaire** : un produit scalaire de Hilbert–Schmidt entre un vecteur de features $\rho(x)$ et un vecteur de poids $M_\theta$. Toute la non-linéarité en $x$ vient de l'encodage ; l'entraînement ne fait que choisir une direction $M_\theta$ dans l'espace des observables. C'est exactement la structure d'une méthode à noyau avec $k(x,x') = \operatorname{Tr}[\rho(x)\rho(x')]$ — on y reviendra en partie 6.

Trois différences importantes avec un réseau de neurones classique :
1. **La sortie est bornée** par la norme de $O$ : il faut adapter les labels (ici $\pm 1$) ou ajouter un post-traitement.
2. **On n'accède jamais à $f_\theta(x)$ exactement** sur un vrai processeur : on l'estime avec un nombre fini de mesures (*shots*), avec une variance $(1-f^2)/N$ pour $N$ shots.
3. **Pas de rétropropagation sur matériel** : l'état intermédiaire n'est pas lisible sans être détruit (partie 3).

In [ ]:
n_qubits = 2
dev = qml.device("default.qubit", wires=n_qubits)

def encodage(x):
    # S(x) : la feature x_i pilote une rotation R_X sur le qubit i.
    # x[..., i] accepte un point (shape (2,)) ou un batch entier (shape (M, 2)) : broadcasting.
    for i in range(n_qubits):
        qml.RX(x[..., i], wires=i)

def couche_ansatz(theta_l):
    # Une couche de W(θ) : rotation générale sur chaque qubit, puis intrication.
    for i in range(n_qubits):
        qml.Rot(*theta_l[i], wires=i)
    qml.CNOT(wires=[0, 1])

def ansatz(theta):
    for theta_l in theta:
        couche_ansatz(theta_l)

@qml.qnode(dev)
def vqc(x, theta):
    encodage(x)
    ansatz(theta)
    return qml.expval(qml.PauliZ(0))

rng = onp.random.default_rng(SEED)
theta_demo = pnp.array(rng.uniform(0, 2 * onp.pi, size=(2, n_qubits, 3)), requires_grad=True)
x_demo = pnp.array([0.3, -0.7], requires_grad=False)

print(qml.draw(vqc, decimals=2)(x_demo, theta_demo))
print(f"\nf_θ(x) = {float(vqc(x_demo, theta_demo)):.6f}")

Lecture du circuit : chaque feature entre par une rotation $R_X(x_i)$ ; viennent ensuite $L$ couches « rotations générales + CNOT ». La porte `Rot(φ, θ, ω)` $= R_Z(\omega)R_Y(\theta)R_Z(\phi)$ peut représenter n'importe quelle porte à un qubit (à une phase près), donc chaque couche porte $3n$ paramètres.

Vérifions numériquement la formule $f_\theta(x) = \operatorname{Tr}[\rho(x) M_\theta]$ en construisant explicitement les deux matrices.

In [ ]:
@qml.qnode(dev)
def etat_encode(x):
    encodage(x)
    return qml.density_matrix(wires=range(n_qubits))

rho = etat_encode(x_demo)                                                   # ρ(x)
W = qml.matrix(ansatz, wire_order=range(n_qubits))(onp.array(theta_demo))  # W(θ)
O = qml.matrix(qml.PauliZ(0), wire_order=range(n_qubits))                   # Z ⊗ I
M = W.conj().T @ O @ W                                                      # M_θ = W† O W

print(f"Tr[ρ(x) M_θ]               = {onp.trace(rho @ M).real:.6f}")
print(f"f_θ(x) (circuit)           = {float(vqc(x_demo, theta_demo)):.6f}")
print(f"M_θ hermitienne ?          {onp.allclose(M, M.conj().T)}")
print(f"valeurs propres de M_θ     {onp.round(onp.linalg.eigvalsh(M), 6)}")

**Ce qu'on observe.** Les deux calculs coïncident, et les valeurs propres de $M_\theta$ sont toujours $\{-1,-1,+1,+1\}$, celles de $Z\otimes I$. L'entraînement ne change pas le *spectre* de l'observable, il la fait seulement *tourner* : l'ensemble des modèles accessibles est l'orbite unitaire de $O$, restreinte aux rotations que l'ansatz sait produire. Un kernel, lui, peut utiliser n'importe quelle combinaison linéaire $\sum_i \alpha_i \rho(x_i)$ comme « observable » : c'est une famille plus large.

---
## 2. Encodage et séries de Fourier : quelles fonctions un VQC peut-il représenter ?

### 2.1 Théorie

Prenons un encodage par des portes $e^{-i x G}$, où $G$ est le générateur (pour $R_X(x)$, $G = X/2$, de valeurs propres $\pm\tfrac12$). En diagonalisant $G$ et en développant le produit des portes, on montre (Schuld, Sweke & Meyer, 2021) que le modèle s'écrit **toujours** :

$$
f_\theta(x) = \sum_{\omega \in \Omega} c_\omega(\theta)\, e^{i \omega x}, \qquad c_{-\omega} = c_\omega^*
$$

- Le **spectre $\Omega$** (les fréquences accessibles) est fixé **uniquement par l'encodage** : ce sont toutes les différences $\Lambda_k - \Lambda_j$ entre sommes de valeurs propres des générateurs.
- Les **coefficients $c_\omega(\theta)$** dépendent de l'ansatz et de l'observable : c'est ce que l'entraînement ajuste.

Si l'on répète $R_X(x)$ $L$ fois (en série, *data re-uploading*) ou qu'on l'applique en parallèle sur $L$ qubits, les sommes de valeurs propres vont de $-\tfrac L2$ à $\tfrac L2$ par pas de 1, donc :

$$
\Omega = \{-L, \dots, -1, 0, 1, \dots, L\}
$$

**Conséquence directe** : un VQC qui encode chaque feature **une seule fois** par une rotation de Pauli est un polynôme trigonométrique de degré 1 en chaque feature. On peut empiler autant de couches entraînables qu'on veut derrière, il ne pourra jamais représenter une fonction qui oscille plus vite. C'est la motivation du *data re-uploading* (Pérez-Salinas et al., 2020).

### 2.2 Expérience : ajuster une série de Fourier de degré 3

On prend une cible dont on connaît le spectre exact :

$$
g(x) = 0.30\cos(x) - 0.25\sin(2x) + 0.35\cos(3x + 0.5)
$$

et un modèle à **un seul qubit** qui alterne $L$ fois [rotation entraînable → $R_X(x)$], suivi d'une dernière rotation. La théorie prédit que le modèle ne peut capturer que les fréquences $\le L$.

Mieux, on peut prédire l'erreur résiduelle : si le modèle reproduit parfaitement les fréquences $\le L$ et rien d'autre, l'erreur quadratique moyenne est l'énergie des fréquences manquantes. Par Parseval, une composante $a\cos(\omega x + \varphi)$ a une énergie moyenne $a^2/2$.

In [ ]:
dev1 = qml.device("default.qubit", wires=1)

@qml.qnode(dev1)
def modele_serie(x, w):
    # 1 qubit, L = w.shape[0] - 1 encodages en série
    L = w.shape[0] - 1
    for l in range(L):
        qml.Rot(*w[l], wires=0)   # bloc entraînable
        qml.RX(x, wires=0)        # (ré-)encodage de la donnée
    qml.Rot(*w[L], wires=0)
    return qml.expval(qml.PauliZ(0))

def cible(x):
    return 0.30 * onp.cos(x) - 0.25 * onp.sin(2 * x) + 0.35 * onp.cos(3 * x + 0.5)

x_fit = onp.linspace(-onp.pi, onp.pi, 60, endpoint=False)
y_fit = cible(x_fit)
X_fit = pnp.array(x_fit, requires_grad=False)

def ajuster(modele, shapes, n_restarts=3, steps=250, lr=0.1, seed=0):
    # Régression MSE avec Adam ; on garde la meilleure de plusieurs initialisations aléatoires.
    rng = onp.random.default_rng(seed)
    meilleur = (onp.inf, None)
    for _ in range(n_restarts):
        params = [pnp.array(rng.uniform(0, 2 * onp.pi, s), requires_grad=True) for s in shapes]
        opt = qml.AdamOptimizer(lr)
        cout = lambda *p: pnp.mean((modele(X_fit, *p) - y_fit) ** 2)
        for _ in range(steps):
            params = opt.step(cout, *params)
            params = list(params) if len(shapes) > 1 else [params]
        mse = float(cout(*params))
        if mse < meilleur[0]:
            meilleur = (mse, params)
    return meilleur

energie = {1: 0.30**2 / 2, 2: 0.25**2 / 2, 3: 0.35**2 / 2}

resultats_serie = {}
t0 = time.time()
for L in [1, 2, 3, 4]:
    mse, params = ajuster(modele_serie, [(L + 1, 3)], seed=L)
    resultats_serie[L] = (mse, params[0])
    prevu = sum(e for freq, e in energie.items() if freq > L)
    print(f"L = {L} | MSE obtenue = {mse:.5f} | MSE prévue (énergie des fréquences > L) = {prevu:.5f}")
print(f"({time.time() - t0:.0f} s)")

In [ ]:
x_plot = onp.linspace(-onp.pi, onp.pi, 300)
fig, axes = plt.subplots(1, 4, figsize=(16, 3.4), sharey=True)
for ax, (L, (mse, w)) in zip(axes, resultats_serie.items()):
    ax.plot(x_plot, modele_serie(pnp.array(x_plot, requires_grad=False), w), lw=4, alpha=0.5, label="VQC")
    ax.plot(x_plot, cible(x_plot), "k--", lw=1.3, label="cible g(x)")
    ax.set_title(f"L = {L} encodage(s)  —  MSE = {mse:.1e}")
    ax.set_xlabel("x")
axes[0].set_ylabel("f(x)")
axes[0].legend(loc="lower left")
plt.tight_layout()
plt.show()

**Ce qu'on observe.** L'erreur obtenue colle à la prédiction théorique : pour $L=1$, le modèle apprend exactement la composante $0.30\cos x$ et rate le reste ; pour $L=2$, il ne manque que la fréquence 3 ; à partir de $L=3$, l'ajustement est parfait. L'optimiseur n'est pas en cause : c'est le **spectre** qui limite le modèle.

On peut le vérifier directement en calculant la transformée de Fourier des fonctions apprises. Pour montrer que la limite est *structurelle* et pas un artefact de l'entraînement, on regarde aussi le plus grand coefficient observé sur 200 jeux de paramètres tirés au hasard.

In [ ]:
def spectre(f_vals):
    # |c_ω| pour ω = 0, 1, 2, ... à partir d'échantillons équirépartis sur [0, 2π)
    return onp.abs(onp.fft.rfft(f_vals) / len(f_vals))

x_fft = pnp.array(onp.linspace(0, 2 * onp.pi, 32, endpoint=False), requires_grad=False)
fig, axes = plt.subplots(1, 2, figsize=(14, 3.8))

# (a) modèles entraînés vs spectre de la cible
largeur = 0.18
for k, (L, (_, w)) in enumerate(resultats_serie.items()):
    c = spectre(modele_serie(x_fft, w))
    axes[0].bar(onp.arange(7) + (k - 1.5) * largeur, c[:7], width=largeur, label=f"L = {L}")
axes[0].scatter([1, 2, 3], [0.15, 0.125, 0.175], marker="_", s=900, c="k", lw=2, label="cible", zorder=5)
axes[0].set_xlabel("fréquence ω")
axes[0].set_ylabel("|c_ω|")
axes[0].set_title("(a) Spectre des modèles entraînés")
axes[0].legend()

# (b) paramètres aléatoires : plus grand |c_ω| observé sur 200 tirages
rng = onp.random.default_rng(1)
for L in [1, 2, 3, 4]:
    c_max = onp.zeros(9)
    for _ in range(200):
        w = rng.uniform(0, 2 * onp.pi, (L + 1, 3))
        c_max = onp.maximum(c_max, spectre(modele_serie(x_fft, w))[:9])
    axes[1].semilogy(range(9), c_max + 1e-18, "o-", label=f"L = {L}")
axes[1].set_ylim(1e-18, 2)
axes[1].set_xlabel("fréquence ω")
axes[1].set_ylabel("max |c_ω| (200 tirages)")
axes[1].set_title("(b) Paramètres aléatoires : le spectre est borné par L")
axes[1].legend()
plt.tight_layout()
plt.show()

**Ce qu'on observe.** En (b), pour chaque $L$, les coefficients chutent brutalement à $10^{-16}$ (le zéro numérique) au-delà de $\omega = L$, quels que soient les paramètres. L'ansatz règle les amplitudes à l'intérieur du spectre, jamais le spectre lui-même.

### 2.3 Série ou parallèle ?

La théorie dit qu'on obtient le même spectre $\{-3,\dots,3\}$ en encodant $x$ trois fois **en série** sur 1 qubit ou **en parallèle** sur 3 qubits. On échange de la profondeur contre de la largeur. Vérifions-le avec un modèle à 3 qubits : un bloc entraînable intriquant (`StronglyEntanglingLayers`), un unique encodage $R_X(x)$ sur chaque qubit, puis un second bloc entraînable.

In [ ]:
dev3 = qml.device("default.qubit", wires=3)

@qml.qnode(dev3)
def modele_parallele(x, w_avant, w_apres):
    qml.StronglyEntanglingLayers(w_avant, wires=range(3))
    for i in range(3):
        qml.RX(x, wires=i)        # la même donnée, encodée une fois sur chaque qubit
    qml.StronglyEntanglingLayers(w_apres, wires=range(3))
    return qml.expval(qml.PauliZ(0))

t0 = time.time()
mse_par, (w_av, w_ap) = ajuster(modele_parallele, [(2, 3, 3), (2, 3, 3)], n_restarts=2, steps=250, seed=7)
print(f"({time.time() - t0:.0f} s)\n")
print(qml.draw(modele_parallele, decimals=None, show_matrices=False)(0.5, w_av, w_ap))

comparaison = pd.DataFrame([
    {"stratégie": "série (re-uploading)", "qubits": 1, "encodages par qubit": 3,
     "paramètres": resultats_serie[3][1].size, "MSE": resultats_serie[3][0]},
    {"stratégie": "parallèle", "qubits": 3, "encodages par qubit": 1,
     "paramètres": w_av.size + w_ap.size, "MSE": mse_par},
])
comparaison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 3.6))

# (a) modèles entraînés
c_serie = spectre(modele_serie(x_fft, resultats_serie[3][1]))[:7]
c_par = spectre(modele_parallele(x_fft, w_av, w_ap))[:7]
axes[0].bar(onp.arange(7) - 0.2, c_serie, width=0.4, label="série, 1 qubit")
axes[0].bar(onp.arange(7) + 0.2, c_par, width=0.4, label="parallèle, 3 qubits")
axes[0].scatter([1, 2, 3], [0.15, 0.125, 0.175], marker="_", s=900, c="k", lw=2, label="cible", zorder=5)
axes[0].set_xlabel("fréquence ω"); axes[0].set_ylabel("|c_ω|")
axes[0].set_title("(a) Modèles entraînés")
axes[0].legend()

# (b) paramètres aléatoires : le spectre accessible est le même
rng = onp.random.default_rng(3)
max_serie, max_par = onp.zeros(9), onp.zeros(9)
for _ in range(100):
    max_serie = onp.maximum(max_serie, spectre(modele_serie(x_fft, rng.uniform(0, 2 * onp.pi, (4, 3))))[:9])
    wa_r, wb_r = rng.uniform(0, 2 * onp.pi, (2, 3, 3)), rng.uniform(0, 2 * onp.pi, (2, 3, 3))
    max_par = onp.maximum(max_par, spectre(modele_parallele(x_fft, wa_r, wb_r))[:9])
axes[1].semilogy(range(9), max_serie + 1e-18, "o-", label="série, 1 qubit, L = 3")
axes[1].semilogy(range(9), max_par + 1e-18, "s--", label="parallèle, 3 qubits")
axes[1].set_ylim(1e-18, 2)
axes[1].set_xlabel("fréquence ω"); axes[1].set_ylabel("max |c_ω| (100 tirages)")
axes[1].set_title("(b) Paramètres aléatoires : même spectre {-3, …, 3}")
axes[1].legend()
plt.tight_layout()
plt.show()

**Ce qu'on observe et ce qu'il faut en retenir.**

- Les deux stratégies atteignent le même spectre et ajustent la cible. La version série est plus économe ici (1 qubit, 12 paramètres) ; la version parallèle est moins profonde, ce qui compte sur du matériel bruité où la décohérence pénalise les circuits longs.
- **Le spectre doit être prévu à la conception.** Si les données ont des composantes de haute fréquence, il faut plus de répétitions, ou bien remettre à l'échelle les features : $R_X(\lambda x)$ produit les fréquences $\lambda\omega$. C'est pour ça que la normalisation des features n'est pas un détail en QML : elle fixe la « bande passante » du modèle.
- **Avoir accès à une fréquence ne garantit pas de pouvoir régler son coefficient librement** : il faut que l'ansatz ait assez de degrés de liberté entre les encodages (ici, des rotations générales à 3 angles).
- En dimension $d$, le nombre de coefficients à contrôler croît comme $(2L+1)^d$ alors que le nombre de paramètres croît linéairement : en grande dimension, c'est l'ansatz qui devient le goulot d'étranglement, pas l'encodage.

---
## 3. Gradients : la *parameter-shift rule*

### 3.1 Théorie

Sur un processeur quantique, on ne peut pas faire de rétropropagation : il faudrait stocker l'état après chaque porte, or le lire le détruit. Il faut donc un moyen d'obtenir $\partial f / \partial\theta_k$ **uniquement à partir d'évaluations du circuit**.

La partie 2 donne la clé. Fixons tous les paramètres sauf un, $\theta_k$, porté par une porte $e^{-i\theta_k P/2}$ avec $P$ un Pauli. C'est exactement la situation de l'encodage : le générateur $P/2$ a pour valeurs propres $\pm\tfrac12$, donc $\theta_k$ n'introduit que les fréquences $\{-1, 0, 1\}$ et

$$
f(\theta_k) = a + b\cos\theta_k + c\sin\theta_k .
$$

La dérivée d'une sinusoïde s'obtient **exactement** par une différence de deux évaluations décalées :

$$
\frac{\partial f}{\partial \theta_k} = \frac{f(\theta_k + s) - f(\theta_k - s)}{2\sin s}
\quad\xrightarrow{\; s = \pi/2 \;}\quad
\frac{f(\theta_k + \tfrac{\pi}{2}) - f(\theta_k - \tfrac{\pi}{2})}{2}
$$

Ce n'est **pas** une approximation de type différences finies : le décalage est macroscopique et le résultat exact (Mitarai et al., 2018 ; Schuld et al., 2019). Pour le gradient complet, on applique la règle à chaque paramètre : **$2P$ évaluations du circuit pour $P$ paramètres**, et ce pour chaque point de données.

### 3.2 Vérifications

D'abord, visualisons que $f$ est bien une sinusoïde en chaque paramètre : on la reconstruit à partir de **trois** évaluations seulement, $f(0)$, $f(\pi/2)$ et $f(\pi)$.

In [ ]:
theta0 = onp.array(theta_demo)
idx = (0, 0, 1)                         # couche 0, qubit 0, angle central du Rot

def f_theta(th):
    return float(vqc(x_demo, pnp.array(th)))

def f_1d(t):
    th = theta0.copy(); th[idx] = t
    return f_theta(th)

t_grid = onp.linspace(-onp.pi, onp.pi, 120)
f_exact = [f_1d(t) for t in t_grid]

f0, f90, f180 = f_1d(0), f_1d(onp.pi / 2), f_1d(onp.pi)
a, b = (f0 + f180) / 2, (f0 - f180) / 2
c = f90 - a

fig, ax = plt.subplots(figsize=(7, 3.4))
ax.plot(t_grid, f_exact, lw=4, alpha=0.4, label="f(θ_k) simulée (120 points)")
ax.plot(t_grid, a + b * onp.cos(t_grid) + c * onp.sin(t_grid), "k--", label="a + b cos θ + c sin θ (3 points)")
ax.scatter([0, onp.pi / 2, onp.pi], [f0, f90, f180], c="crimson", zorder=5, label="3 évaluations")
ax.set_xlabel("θ_k"); ax.set_ylabel("f")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

Comparons maintenant quatre façons de calculer le gradient complet (12 paramètres) : la *parameter-shift rule*, l'autodifférentiation du simulateur (backprop), et des différences finies centrées avec trois valeurs de $\varepsilon$.

In [ ]:
def parameter_shift(fonction, theta, k, s=onp.pi / 2):
    tp, tm = theta.copy(), theta.copy()
    tp[k] += s; tm[k] -= s
    return (fonction(tp) - fonction(tm)) / (2 * onp.sin(s))

def diff_finies(fonction, theta, k, eps):
    tp, tm = theta.copy(), theta.copy()
    tp[k] += eps; tm[k] -= eps
    return (fonction(tp) - fonction(tm)) / (2 * eps)

grad_ref = onp.array(qml.grad(lambda th: vqc(x_demo, th))(theta_demo))   # backprop (référence)
grad_ps = onp.zeros_like(theta0)
grad_ps_s = onp.zeros_like(theta0)
for k in onp.ndindex(theta0.shape):
    grad_ps[k] = parameter_shift(f_theta, theta0, k)
    grad_ps_s[k] = parameter_shift(f_theta, theta0, k, s=0.9)        # décalage quelconque

lignes = [
    {"méthode": "parameter-shift (s = π/2)", "évaluations": 2 * theta0.size, "erreur max": onp.max(onp.abs(grad_ps - grad_ref))},
    {"méthode": "parameter-shift (s = 0.9)", "évaluations": 2 * theta0.size, "erreur max": onp.max(onp.abs(grad_ps_s - grad_ref))},
]
for eps in [1e-1, 1e-4, 1e-9]:
    g = onp.array([diff_finies(f_theta, theta0, k, eps) for k in onp.ndindex(theta0.shape)]).reshape(theta0.shape)
    lignes.append({"méthode": f"différences finies (ε = {eps:.0e})", "évaluations": 2 * theta0.size,
                   "erreur max": onp.max(onp.abs(g - grad_ref))})
pd.DataFrame(lignes).style.format({"erreur max": "{:.1e}"})

**Ce qu'on observe.** La *parameter-shift rule* est exacte à la précision machine ($\sim 10^{-16}$), quel que soit le décalage $s$. Les différences finies, elles, font face au dilemme classique : $\varepsilon$ trop grand donne une erreur de troncature en $\varepsilon^2$, $\varepsilon$ trop petit amplifie les erreurs d'arrondi.

Mais en simulation exacte, les différences finies restent acceptables avec un $\varepsilon$ bien choisi. La vraie raison d'utiliser la *parameter-shift rule* apparaît quand on simule ce qui se passe sur du matériel : **le bruit de mesure**.

### 3.3 Le vrai juge : le bruit de shots

Sur un processeur, $f$ est la moyenne de $N$ mesures valant $\pm 1$. L'estimateur a une variance $(1-f^2)/N$. Pour une différence finie, cette variance est divisée par $(2\varepsilon)^2$ : elle **explose** quand $\varepsilon \to 0$. Pour la *parameter-shift rule*, le dénominateur vaut 2, quelle que soit la précision souhaitée.

On simule ce bruit exactement : avec $p_+ = (1+f)/2$, le nombre de résultats $+1$ suit une loi binomiale $\mathcal{B}(N, p_+)$. On répète 5 000 estimations du gradient avec $N = 1000$ shots par évaluation, et on décompose l'erreur quadratique moyenne en biais² + variance.

In [ ]:
def estimer_expval(f_exacte, n_shots, n_rep, rng):
    # n_rep estimations de <Z> avec n_shots mesures ±1 chacune
    p_plus = (1 + f_exacte) / 2
    return 2 * rng.binomial(n_shots, p_plus, size=n_rep) / n_shots - 1

N_SHOTS, N_REP = 1000, 5000
g_vrai = grad_ref[idx]
rng = onp.random.default_rng(0)

def analyse_estimateur(decalage, denom):
    # Renvoie (biais² exact, variance théorique, MSE empirique) pour l'estimateur (f(θ+s) - f(θ-s)) / denom
    tp, tm = theta0.copy(), theta0.copy()
    tp[idx] += decalage; tm[idx] -= decalage
    f_p, f_m = f_theta(tp), f_theta(tm)
    biais2 = ((f_p - f_m) / denom - g_vrai) ** 2                              # erreur systématique
    variance = ((1 - f_p**2) + (1 - f_m**2)) / (N_SHOTS * denom**2)           # bruit de shots
    est = (estimer_expval(f_p, N_SHOTS, N_REP, rng) - estimer_expval(f_m, N_SHOTS, N_REP, rng)) / denom
    return biais2, variance, onp.mean((est - g_vrai) ** 2)

epsilons = onp.logspace(-3, 0.2, 30)
fd = onp.array([analyse_estimateur(e, 2 * e) for e in epsilons])     # colonnes : biais², variance, MSE empirique
ps_biais2, ps_var, ps_mse = analyse_estimateur(onp.pi / 2, 2.0)

fig, ax = plt.subplots(figsize=(8, 4))
ax.loglog(epsilons, fd[:, 2], "o", c="C0", label="différences finies : MSE mesurée (5 000 répétitions)")
ax.loglog(epsilons, fd[:, 0] + fd[:, 1], "-", c="C0", alpha=0.7, label="   théorie : biais² + variance")
ax.loglog(epsilons, fd[:, 1], ":", c="C0", label="   dont variance (shots) ∝ 1/ε²")
ax.loglog(epsilons, fd[:, 0], "--", c="C0", alpha=0.6, label="   dont biais² ∝ ε⁴")
ax.axhline(ps_mse, c="crimson", lw=2, label="parameter-shift : MSE mesurée")
ax.set_xlabel("ε"); ax.set_ylabel("erreur quadratique moyenne sur ∂f/∂θ")
ax.set_ylim(1e-6, 1e3)
ax.set_title(f"Estimation d'une dérivée avec {N_SHOTS} shots par évaluation")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

i_best = onp.argmin(fd[:, 2])
print(f"Meilleure différence finie : ε ≈ {epsilons[i_best]:.2f}, MSE = {fd[i_best, 2]:.2e}")
print(f"Parameter-shift            : MSE = {ps_mse:.2e} (biais² exact = {ps_biais2:.1e})")

**Ce qu'on observe.** Les mesures (points) suivent exactement la décomposition théorique (traits). La courbe des différences finies a la forme en « V » typique d'un compromis biais–variance : à gauche la variance due aux shots explose en $1/\varepsilon^2$, à droite le biais de troncature (en $\varepsilon^4$ pour l'erreur quadratique) prend le dessus. Même au meilleur $\varepsilon$, l'erreur reste au-dessus de celle de la *parameter-shift rule*, qui est **sans biais** et a une variance minimale. On ne peut pas « améliorer » une différence finie en réduisant $\varepsilon$ sans payer en shots, alors que la *parameter-shift rule* n'a pas de paramètre à régler.

### 3.4 Le prix à payer : le nombre d'exécutions

Comparons, sur un circuit à 4 qubits de profondeur croissante, le nombre d'exécutions de circuit et le temps de calcul pour obtenir un gradient complet avec trois méthodes :
- **parameter-shift** : la seule utilisable sur matériel ;
- **backprop** : autodifférentiation à travers le simulateur (impossible sur matériel) ;
- **adjoint** : méthode propre aux simulateurs qui « rembobine » l'état, très économe en mémoire.

In [ ]:
dev4 = qml.device("default.qubit", wires=4)

def circuit_profond(diff_method):
    @qml.qnode(dev4, diff_method=diff_method)
    def c(w):
        qml.StronglyEntanglingLayers(w, wires=range(4))
        return qml.expval(qml.PauliZ(0))
    return c

lignes = []
for L in [1, 2, 4, 8, 16]:
    w = pnp.array(onp.random.default_rng(L).uniform(0, 2 * onp.pi, (L, 4, 3)), requires_grad=True)
    for methode in ["parameter-shift", "backprop", "adjoint"]:
        c = circuit_profond(methode)
        qml.grad(c)(w)                                      # échauffement
        with qml.Tracker(dev4) as tracker:
            t0 = time.perf_counter()
            qml.grad(c)(w)
            dt = time.perf_counter() - t0
        lignes.append({"P": w.size, "méthode": methode,
                       "exécutions": tracker.totals["executions"], "temps (ms)": 1000 * dt})
df_cout = pd.DataFrame(lignes)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
for methode, grp in df_cout.groupby("méthode"):
    axes[0].plot(grp["P"], grp["exécutions"], "o-", label=methode)
    axes[1].plot(grp["P"], grp["temps (ms)"], "o-", label=methode)
axes[0].set_xscale("log"); axes[0].set_yscale("log")
axes[0].set_xlabel("nombre de paramètres P"); axes[0].set_ylabel("exécutions de circuit")
axes[0].set_title("Exécutions par gradient")
axes[1].set_xscale("log"); axes[1].set_yscale("log")
axes[1].set_xlabel("nombre de paramètres P"); axes[1].set_ylabel("temps (ms)")
axes[1].set_title("Temps par gradient (simulateur)")
axes[0].legend()
plt.tight_layout()
plt.show()
tableau = df_cout.pivot(index="P", columns="méthode", values="exécutions")
tableau["2P + 1 (théorie PS)"] = 2 * tableau.index + 1
tableau

**Ce qu'on observe.** La *parameter-shift rule* demande $2P + 1$ exécutions (la $+1$ est l'évaluation de $f$ elle-même), alors que backprop et adjoint n'en demandent qu'une, quel que soit $P$.

Détail intéressant : pour $P = 24$ et $P = 96$, PennyLane fait **moins** que $2P+1$ exécutions. En analysant le graphe du circuit, il détecte que 6 paramètres de la dernière couche ne peuvent pas influencer $\langle Z_0\rangle$ : aucune chaîne de portes à deux qubits placée *après* elles ne relie leur qubit au qubit 0. Leur dérivée est donc nulle sans calcul. C'est une première apparition du **cône de causalité** de l'observable, une notion qui sera centrale dans les parties 4 et 5. (Les temps mesurés sont indicatifs et un peu bruités.) Sur simulateur, on utilise donc toujours backprop ou adjoint ; mais ces méthodes n'existent pas sur un processeur réel.

**Ordre de grandeur sur matériel.** Une époque d'entraînement coûte $M \times (2P+1) \times N_\text{shots}$ mesures. Avec $M = 1000$ exemples, $P = 100$ paramètres et $N_\text{shots} = 1000$, on arrive à $2\times 10^8$ mesures *par époque*. Ce coût, plus que la qualité des modèles, est souvent ce qui limite les VQC en pratique. Des estimateurs stochastiques comme SPSA (2 évaluations par gradient quelle que soit la valeur de $P$) échangent ce coût contre davantage de bruit.

---
## 4. Entraîner un VQC sur un vrai problème de classification

### 4.1 Le problème

On utilise le dataset *two moons* (200 points, bruit 0.15), séparé en 70 % d'entraînement et 30 % de test. Les features sont remises à l'échelle dans $[-\pi/2, \pi/2]$ (la « bande passante » discutée en partie 2), et les labels sont dans $\{-1, +1\}$ pour correspondre à la plage de $\langle Z\rangle$.

### 4.2 Deux architectures, trois profondeurs

On compare deux familles, chacune avec $L \in \{1, 3, 5\}$ couches :

- **A — encodage unique** : $S(x)$ une fois, puis $L$ couches d'ansatz. D'après la partie 2, c'est un polynôme trigonométrique de **degré 1** en chaque feature, quel que soit $L$.
- **B — re-uploading** : $L$ fois [$S(x)$ → couche d'ansatz]. Degré **$L$** en chaque feature.

Le modèle de classification est $\hat y = \operatorname{signe}\big(f_\theta(x) + b\big)$ avec un biais classique $b$ entraînable. On minimise l'erreur quadratique $\frac1M \sum_i (f_\theta(x_i) + b - y_i)^2$ avec Adam (pas de 0.05, 150 itérations, batch complet).

In [ ]:
X_all, y_all = make_moons(n_samples=200, noise=0.15, random_state=SEED)
y_all = 2 * y_all - 1
X_tr_brut, X_te_brut, y_tr, y_te = train_test_split(X_all, y_all, test_size=0.3, random_state=SEED, stratify=y_all)

scaler = MinMaxScaler(feature_range=(-onp.pi / 2, onp.pi / 2)).fit(X_tr_brut)
X_tr, X_te = scaler.transform(X_tr_brut), scaler.transform(X_te_brut)
X_tr_q = pnp.array(X_tr, requires_grad=False)
X_te_q = pnp.array(X_te, requires_grad=False)

fig, ax = plt.subplots(figsize=(5, 3.8))
ax.scatter(X_tr[:, 0], X_tr[:, 1], c=y_tr, cmap="RdBu", edgecolors="k", s=25, label="train")
ax.scatter(X_te[:, 0], X_te[:, 1], c=y_te, cmap="RdBu", edgecolors="k", marker="^", s=40, label="test")
ax.set_xlabel("x₁ (→ qubit 0)"); ax.set_ylabel("x₂ (→ qubit 1)")
ax.legend()
plt.tight_layout()
plt.show()
print(f"train : {len(y_tr)} points | test : {len(y_te)} points")

In [ ]:
@qml.qnode(dev)
def vqc_unique(x, theta):              # architecture A
    encodage(x)
    for theta_l in theta:
        couche_ansatz(theta_l)
    return qml.expval(qml.PauliZ(0))

@qml.qnode(dev)
def vqc_reupload(x, theta):            # architecture B
    for theta_l in theta:
        encodage(x)
        couche_ansatz(theta_l)
    return qml.expval(qml.PauliZ(0))

def precision(modele, theta, b, X, y):
    return float(onp.mean(onp.sign(modele(X, theta) + b) == y))

def entrainer_vqc(modele, L, steps=150, lr=0.05, seed=0):
    rng = onp.random.default_rng(seed)
    theta = pnp.array(rng.uniform(0, 2 * onp.pi, (L, n_qubits, 3)), requires_grad=True)
    b = pnp.array(0.0, requires_grad=True)
    opt = qml.AdamOptimizer(lr)
    cout = lambda theta, b: pnp.mean((modele(X_tr_q, theta) + b - y_tr) ** 2)
    hist = {"loss": [], "acc_tr": [], "acc_te": []}
    for _ in range(steps):
        (theta, b), loss = opt.step_and_cost(cout, theta, b)
        hist["loss"].append(float(loss))
        hist["acc_tr"].append(precision(modele, theta, b, X_tr_q, y_tr))
        hist["acc_te"].append(precision(modele, theta, b, X_te_q, y_te))
    return theta, b, hist

architectures = {"A — encodage unique": vqc_unique, "B — re-uploading": vqc_reupload}
resultats_vqc = {}
t0 = time.time()
for nom, modele in architectures.items():
    for L in [1, 3, 5]:
        theta, b, hist = entrainer_vqc(modele, L)
        resultats_vqc[(nom, L)] = {"modele": modele, "theta": theta, "b": b, "hist": hist}
print(f"6 entraînements en {time.time() - t0:.0f} s")

pd.DataFrame([
    {"architecture": nom, "L": L, "paramètres": r["theta"].size + 1,
     "loss finale": r["hist"]["loss"][-1],
     "précision train": precision(r["modele"], r["theta"], r["b"], X_tr_q, y_tr),
     "précision test": precision(r["modele"], r["theta"], r["b"], X_te_q, y_te)}
    for (nom, L), r in resultats_vqc.items()
])

In [ ]:
# Outils de visualisation des frontières de décision (réutilisés en partie 6)
marge = 0.3
xx, yy = onp.meshgrid(onp.linspace(-onp.pi / 2 - marge, onp.pi / 2 + marge, 70),
                      onp.linspace(-onp.pi / 2 - marge, onp.pi / 2 + marge, 70))
grille = onp.c_[xx.ravel(), yy.ravel()]
grille_q = pnp.array(grille, requires_grad=False)

def tracer_frontiere(ax, scores, titre):
    s = onp.asarray(scores).reshape(xx.shape)
    s = s / (onp.max(onp.abs(s)) + 1e-12)                # normalisation pour comparer les modèles
    ax.contourf(xx, yy, s, levels=onp.linspace(-1, 1, 21), cmap="RdBu", alpha=0.7)
    ax.contour(xx, yy, s, levels=[0], colors="k", linewidths=1.5)
    ax.scatter(X_tr[:, 0], X_tr[:, 1], c=y_tr, cmap="RdBu", edgecolors="k", s=14)
    ax.scatter(X_te[:, 0], X_te[:, 1], c=y_te, cmap="RdBu", edgecolors="k", marker="^", s=22)
    ax.set_title(titre, fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])

fig, axes = plt.subplots(2, 3, figsize=(13, 8.2))
for ax, ((nom, L), r) in zip(axes.ravel(), resultats_vqc.items()):
    scores = r["modele"](grille_q, r["theta"]) + r["b"]
    acc = precision(r["modele"], r["theta"], r["b"], X_te_q, y_te)
    tracer_frontiere(ax, scores, f"{nom}, L = {L}\nprécision test = {acc:.0%}")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
for (nom, L), r in resultats_vqc.items():
    style = "-" if nom.startswith("A") else "--"
    couleur = {1: "C0", 3: "C1", 5: "C2"}[L]
    axes[0].plot(r["hist"]["loss"], style, c=couleur, label=f"{nom[:1]}, L = {L}")
    axes[1].plot(r["hist"]["acc_te"], style, c=couleur, label=f"{nom[:1]}, L = {L}")
axes[0].set_xlabel("itération"); axes[0].set_ylabel("loss (MSE)"); axes[0].set_title("Loss d'entraînement")
axes[1].set_xlabel("itération"); axes[1].set_ylabel("précision"); axes[1].set_title("Précision sur le test")
axes[0].legend(ncol=2, fontsize=8)
plt.tight_layout()
plt.show()

**Ce qu'on observe — trois enseignements.**

**1. Le piège du cône de causalité ($L = 1$).** Avec une seule couche, les deux architectures sont identiques et plafonnent autour de 70 % avec une frontière **verticale** : le modèle ne dépend que de $x_1$ ! En représentation de Heisenberg, l'observable mesurée $Z_0$ traverse le circuit à l'envers : le CNOT (contrôle 0, cible 1) laisse $Z_0$ invariant, puis seule la rotation du qubit 0 agit. Le qubit 1, et donc $x_2$, n'atteint jamais la mesure. Avant d'entraîner un circuit, il faut vérifier que chaque feature est dans le *cône de causalité* de l'observable — c'est la même notion qui reviendra en partie 5.

**2. Le plafond d'expressivité de l'encodage unique.** L'architecture A atteint pratiquement la même loss pour $L = 3$ et $L = 5$ (0.434 contre 0.433) : les 12 paramètres supplémentaires sont inutiles. Le modèle est un polynôme trigonométrique de degré 1 en chaque feature (9 coefficients de Fourier au total), et $L = 3$ suffit déjà à les régler au mieux. La frontière reste une forme « douce » incapable de suivre l'imbrication des deux lunes.

**3. Le re-uploading lève la limite.** Avec B, la loss diminue nettement avec $L$ et la précision de test dépasse 95 %. Chaque couche supplémentaire ajoute une fréquence par feature, ce qui permet à la frontière de se courber. La contrepartie existe : plus de fréquences, c'est aussi plus de capacité à sur-apprendre le bruit sur de petits datasets — $L$ joue le rôle du degré d'un polynôme et se règle comme un hyperparamètre de régularisation.

---
## 5. Le grand problème : les *barren plateaus*

### 5.1 Théorie

Jusqu'ici, tout fonctionne parce que nos circuits ont 1 à 3 qubits. Que se passe-t-il quand on passe à l'échelle ?

**Résultat central (McClean et al., 2018).** Si l'ansatz $W(\theta)$ avec des paramètres aléatoires est suffisamment « mélangeant » (techniquement, s'il forme une *2-design* unitaire, ce qui arrive pour des circuits profonds), alors pour tout paramètre $\theta_k$ :

$$
\mathbb{E}_\theta\!\left[\partial_k C\right] = 0
\qquad\text{et}\qquad
\operatorname{Var}_\theta\!\left[\partial_k C\right] \in \mathcal{O}\!\left(2^{-n}\right)
$$

Le paysage de la fonction de coût devient **exponentiellement plat** avec le nombre de qubits $n$. Ce n'est pas un problème de minimum local : c'est le paysage entier qui se concentre autour de sa moyenne.

**Pourquoi c'est rédhibitoire.** Avec $N$ shots, l'erreur statistique sur une dérivée est de l'ordre de $1/\sqrt{N}$. Pour distinguer un gradient d'amplitude $2^{-n/2}$ du bruit, il faut $N \gtrsim 2^{n}$ shots. Les optimiseurs sans gradient n'y échappent pas non plus : ils comparent des valeurs de coût qui sont elles aussi exponentiellement concentrées.

**Les facteurs aggravants** identifiés depuis : l'expressivité de l'ansatz (Holmes et al., 2022), le caractère **global** de la fonction de coût (Cerezo et al., 2021), l'intrication produite par l'encodage, et le bruit matériel, qui provoque des plateaux même pour des circuits peu profonds (Wang et al., 2021).

### 5.2 Le dispositif expérimental

On utilise un ansatz « hardware-efficient » classique : une couche initiale $R_Y(\pi/4)$, puis $L$ couches de [$R_X$, $R_Y$ sur chaque qubit + une échelle de CZ]. Pour chaque taille $n$, on tire 300 jeux de paramètres uniformes dans $[0, 2\pi)$ et on calcule la dérivée par rapport au **premier** paramètre avec la *parameter-shift rule*. Grâce au broadcasting, les 300 circuits sont simulés d'un coup.

On mesure l'amplitude typique du gradient $\mathbb{E}[(\partial_k C)^2]$ (égale à la variance quand la moyenne est nulle).

In [ ]:
def creer_circuit(n, L, observable):
    dev_n = qml.device("default.qubit", wires=n)

    @qml.qnode(dev_n, diff_method=None)
    def circuit(params):                     # params : shape (L, n, 2, S) → S circuits simulés en parallèle
        for i in range(n):
            qml.RY(onp.pi / 4, wires=i)
        for l in range(L):
            for i in range(n):
                qml.RX(params[l, i, 0], wires=i)
                qml.RY(params[l, i, 1], wires=i)
            for i in range(n - 1):
                qml.CZ(wires=[i, i + 1])
        if observable == "global":           # Z ⊗ Z ⊗ ... ⊗ Z : agit sur tous les qubits
            return qml.expval(reduce(lambda a, b: a @ b, [qml.PauliZ(i) for i in range(n)]))
        if observable == "local":            # Z sur un seul qubit
            return qml.expval(qml.PauliZ(0))
        return qml.expval(qml.PauliZ(0) @ qml.PauliZ(1))   # "zz" : 2 qubits voisins (McClean et al.)

    return circuit

def gradients_aleatoires(n, L, observable, n_samples=300, init="uniforme", seed=0):
    rng = onp.random.default_rng(seed)
    if init == "uniforme":
        p = rng.uniform(0, 2 * onp.pi, (L, n, 2, n_samples))
    else:                                    # "petits angles" : circuit proche de l'identité
        p = rng.normal(0, 0.1, (L, n, 2, n_samples))
    circuit = creer_circuit(n, L, observable)
    p_plus, p_moins = p.copy(), p.copy()
    p_plus[0, 0, 0] += onp.pi / 2
    p_moins[0, 0, 0] -= onp.pi / 2
    return (circuit(p_plus) - circuit(p_moins)) / 2

### 5.3 À quoi ressemble un paysage plat ?

Avant les statistiques, une intuition visuelle : on fixe un point aléatoire $\theta_0$ d'un circuit profond ($L = 40$), on tire quelques directions aléatoires $d$ (composantes $d_k \sim \mathcal{N}(0,1)$) et on trace $C(\theta_0 + t\,d)$ pour $t \in [-1, 1]$. Chaque paramètre se déplace donc d'environ $t$ radians, **quel que soit $n$** : la comparaison entre tailles est équitable.

In [ ]:
L_PROFOND = 40
fig, axes = plt.subplots(1, 3, figsize=(14, 3.4), sharey=True)
t_vals = onp.linspace(-1, 1, 150)
for ax, n in zip(axes, [2, 6, 10]):
    rng = onp.random.default_rng(n)
    circuit = creer_circuit(n, L_PROFOND, "zz")
    theta_0 = rng.uniform(0, 2 * onp.pi, (L_PROFOND, n, 2))
    for _ in range(4):
        d = rng.normal(size=theta_0.shape)
        params = theta_0[..., None] + t_vals * d[..., None]    # broadcasting : 150 points d'un coup
        ax.plot(t_vals, circuit(params), lw=1.3)
    ax.set_title(f"n = {n} qubits ({L_PROFOND * n * 2} paramètres)")
    ax.set_xlabel("t")
axes[0].set_ylabel("C(θ₀ + t·d) = ⟨Z₀Z₁⟩")
plt.suptitle("Coupes 1D du paysage d'un circuit profond (L = 40) : il se concentre autour de 0 quand n augmente", y=1.03)
plt.tight_layout()
plt.show()

### 5.4 Décroissance exponentielle des gradients (circuit profond)

In [ ]:
tailles = [2, 3, 4, 5, 6, 7, 8, 9, 10]
t0 = time.time()
g2_profond = {n: onp.mean(gradients_aleatoires(n, L_PROFOND, "zz") ** 2) for n in tailles}
print(f"({time.time() - t0:.0f} s)")

ns = onp.array(tailles)
vals = onp.array([g2_profond[n] for n in tailles])
pente, ordonnee = onp.polyfit(ns, onp.log2(vals), 1)

fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy(ns, vals, "o-", lw=2, label="E[(∂C)²] mesurée")
ax.semilogy(ns, 2.0 ** (ordonnee + pente * ns), "k--", label=f"ajustement : 2^({pente:.2f}·n)")
ax.set_xlabel("nombre de qubits n"); ax.set_ylabel("E[(∂C)²]")
ax.set_title(f"Ansatz hardware-efficient, L = {L_PROFOND}, observable Z₀Z₁")
ax.legend()
plt.tight_layout()
plt.show()
print(f"Facteur de décroissance par qubit ajouté : ×{2 ** pente:.2f}")

**Ce qu'on observe.** L'amplitude du gradient est divisée par environ 2 à chaque qubit ajouté, exactement la loi en $2^{-n}$ prédite pour une 2-design. Les coupes 1D le montraient déjà : à $n = 10$, les variations du coût sont minuscules par rapport à $n = 2$.

On peut traduire ça en coût expérimental. Pour un rapport signal/bruit de 1 avec la *parameter-shift rule*, il faut environ $N \approx 1/(2\,\mathbb{E}[(\partial C)^2])$ shots par évaluation. Extrapolons avec la pente ajustée :

In [ ]:
def shots_necessaires(n):
    return 1 / (2 * 2.0 ** (ordonnee + pente * n))

pd.DataFrame([{"qubits": n, "E[(∂C)²] (extrapolée)": 2.0 ** (ordonnee + pente * n),
               "shots par évaluation (SNR ≈ 1)": f"{shots_necessaires(n):.1e}"}
              for n in [10, 20, 30, 50, 100]])

À 50 qubits, il faudrait de l'ordre de $10^{15}$ mesures **pour une seule composante d'un seul gradient**. Un circuit profond initialisé au hasard est donc inentraînable à grande échelle, même sur une machine parfaite sans bruit.

### 5.5 Le rôle de la fonction de coût : global vs local

Cerezo et al. (2021) montrent que la profondeur n'est pas la seule cause : une **observable globale** (qui agit sur tous les qubits) crée un plateau même pour un circuit **peu profond**, alors qu'une observable **locale** sur un circuit peu profond n'en crée pas. On compare, avec $L = 2$ :
- **global** : $C = \langle Z_0 Z_1 \cdots Z_{n-1}\rangle$ ;
- **local** : $C = \langle Z_0 \rangle$.

### 5.6 Une piste d'atténuation : l'initialisation

Le théorème suppose des paramètres uniformes. Si l'on initialise les angles près de zéro, $\theta \sim \mathcal{N}(0, 0.1^2)$, le circuit profond démarre proche de l'identité, loin du régime 2-design (idée proche de Grant et al., 2019).

In [ ]:
t0 = time.time()
g2_global = {n: onp.mean(gradients_aleatoires(n, 2, "global") ** 2) for n in tailles}
g2_local = {n: onp.mean(gradients_aleatoires(n, 2, "local") ** 2) for n in tailles}
g2_petit = {n: onp.mean(gradients_aleatoires(n, L_PROFOND, "zz", init="petits angles") ** 2) for n in tailles}
print(f"({time.time() - t0:.0f} s)")

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
axes[0].semilogy(tailles, [g2_local[n] for n in tailles], "o-", lw=2, label="L = 2, observable locale Z₀")
axes[0].semilogy(tailles, [g2_global[n] for n in tailles], "s-", lw=2, label="L = 2, observable globale Z⊗…⊗Z")
axes[0].semilogy(tailles, [g2_profond[n] for n in tailles], "^--", c="gray", label=f"L = {L_PROFOND}, Z₀Z₁ (rappel)")
axes[0].set_title("(a) Profondeur et localité de la fonction de coût")
axes[0].set_xlabel("nombre de qubits n"); axes[0].set_ylabel("E[(∂C)²]")
axes[0].legend(fontsize=8)

axes[1].semilogy(tailles, [g2_profond[n] for n in tailles], "^-", lw=2, c="gray", label="L = 40, init. uniforme")
axes[1].semilogy(tailles, [g2_petit[n] for n in tailles], "D-", lw=2, c="C2", label="L = 40, init. petits angles")
axes[1].set_title("(b) Initialisation")
axes[1].set_xlabel("nombre de qubits n")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

**Ce qu'on observe.**

- **(a)** Avec un circuit peu profond, l'observable **locale** donne des gradients d'amplitude constante quand $n$ augmente, alors que l'observable **globale** s'effondre exponentiellement, encore plus vite que le circuit profond. L'explication tient au cône de causalité déjà rencontré en partie 4 : avec $L = 2$, $Z_0$ ne « voit » que les quelques qubits voisins, donc la taille effective du problème ne grandit pas avec $n$. Une observable globale, elle, est un produit de $n$ facteurs de module inférieur à 1, ce qui l'écrase exponentiellement.
- **(b)** L'initialisation près de l'identité garde des gradients d'amplitude constante même pour 40 couches.

**Attention aux conclusions trop rapides.** Ces remèdes ont un coût. Un circuit peu profond avec des coûts locaux, ou un circuit qui reste proche de l'identité, explore une petite région de l'espace des états. Plusieurs travaux récents (Cerezo et al., 2023) suggèrent que les architectures pour lesquelles on sait *prouver* l'absence de plateaux sont souvent, pour la même raison, simulables efficacement par un ordinateur classique. Trainabilité et avantage quantique sont en tension, et c'est l'une des grandes questions ouvertes du domaine.

> **Lien avec les kernels.** Le même phénomène existe pour les kernels quantiques sous le nom de **concentration exponentielle** (Thanasilp et al., 2024) : pour un encodage très expressif sur beaucoup de qubits, les valeurs $k(x, x') = |\langle\psi(x)|\psi(x')\rangle|^2$ se concentrent exponentiellement autour d'une constante, la matrice de Gram tend vers l'identité et le modèle ne généralise plus. Si ton notebook sur les kernels n'a pas encore cette expérience, c'est un excellent ajout.

---
## 6. Face-à-face : VQC vs kernel quantique vs modèles classiques

### 6.1 Le pont théorique

On a vu en partie 1 que $f_\theta(x) = \operatorname{Tr}[\rho(x) M_\theta]$. Pour un encodage $\rho(x)$ fixé, une méthode à noyau avec

$$
k(x, x') = \operatorname{Tr}[\rho(x)\rho(x')] = |\langle \psi(x) | \psi(x') \rangle|^2
$$

cherche son modèle sous la forme $f(x) = \operatorname{Tr}[\rho(x) M]$ avec $M = \sum_i \alpha_i\, \rho(x_i)$. Par le **théorème du représentant**, l'optimum du problème régularisé sur **toutes** les observables $M$ est de cette forme : le kernel trouve la meilleure mesure possible pour cet encodage, et le problème d'optimisation est **convexe** (Schuld, 2021).

Le VQC, lui, ne cherche que parmi les observables $M_\theta = W(\theta)^\dagger O W(\theta)$ atteignables par l'ansatz, avec un problème **non convexe**. En contrepartie, son coût d'entraînement est linéaire en nombre de données, contre quadratique pour le kernel.

Nuance importante : avec du re-uploading, l'encodage n'est plus « un $S(x)$ puis un $W(\theta)$ ». Le modèle B de la partie 4 correspond à un encodage différent et plus riche, et n'est pas couvert par la comparaison avec un kernel construit sur $S(x)$.

### 6.2 Les concurrents

| famille | modèle | ce qu'on compare |
|---|---|---|
| classique | régression logistique | référence linéaire |
| classique | SVM noyau RBF | référence non linéaire standard |
| classique | MLP (16, 16) | réseau de neurones de taille comparable |
| kernel quantique | SVM, noyau « angle » | même encodage $S(x)$ que l'architecture A |
| kernel quantique | SVM, noyau IQP (2 répétitions) | encodage intriquant (Havlíček et al., 2019) |
| VQC | architectures A et B ($L = 5$) | entraînés en partie 4 |

In [ ]:
@qml.qnode(dev)
def etat_angle(x):
    encodage(x)                                           # même S(x) que l'architecture A
    return qml.state()

@qml.qnode(dev)
def etat_iqp(x):
    qml.IQPEmbedding(x, wires=range(n_qubits), n_repeats=2)
    return qml.state()

def matrice_gram(etat, XA, XB):
    SA = etat(pnp.array(XA, requires_grad=False))
    SB = etat(pnp.array(XB, requires_grad=False))
    return onp.abs(onp.conj(SA) @ SB.T) ** 2             # |<ψ(a)|ψ(b)>|² pour toutes les paires

# Remarque : le noyau « angle » sans intrication a une forme close classique
K_angle = matrice_gram(etat_angle, X_tr, X_tr)
K_angle_classique = onp.prod(onp.cos((X_tr[:, None, :] - X_tr[None, :, :]) / 2) ** 2, axis=-1)
print("Noyau angle = Π cos²((x_i - x'_i)/2) ?", onp.allclose(K_angle, K_angle_classique))

Premier enseignement avant même de comparer les scores : le noyau issu de l'encodage par angles **sans intrication** est un produit de $\cos^2$, calculable classiquement en $\mathcal{O}(d)$. Un modèle quantique dont le noyau se calcule aussi facilement ne peut offrir aucun avantage. L'intrication dans l'encodage (comme dans IQP) est une condition nécessaire, mais pas suffisante, pour sortir de ce cas.

In [ ]:
lignes, scores_grille = [], {}
M_tr, M_te = len(y_tr), len(y_te)

# --- Modèles classiques ---
classiques = {
    "Régression logistique": LogisticRegression(),
    "SVM RBF": SVC(kernel="rbf", C=1.0, gamma="scale"),
    "MLP (16, 16)": MLPClassifier(hidden_layer_sizes=(16, 16), max_iter=3000, random_state=SEED),
}
def nb_params(clf):
    if hasattr(clf, "coefs_"):                       # MLP : poids + biais
        return sum(w.size for w in clf.coefs_) + sum(b.size for b in clf.intercepts_)
    if isinstance(clf, SVC):                         # SVM : un coefficient dual par vecteur support
        return len(clf.support_)
    return clf.coef_.size + clf.intercept_.size      # modèle linéaire

for nom, clf in classiques.items():
    clf.fit(X_tr, y_tr)
    lignes.append({"modèle": nom, "famille": "classique", "paramètres": nb_params(clf),
                   "précision train": clf.score(X_tr, y_tr), "précision test": clf.score(X_te, y_te),
                   "circuits (entraînement)": 0, "circuits (prédiction test)": 0, "optimisation": "convexe" if nom != "MLP (16, 16)" else "non convexe"})
    scores_grille[nom] = clf.decision_function(grille) if hasattr(clf, "decision_function") else clf.predict_proba(grille)[:, 1] - 0.5

# --- Kernels quantiques ---
for nom, etat in [("Kernel quantique angle", etat_angle), ("Kernel quantique IQP", etat_iqp)]:
    K_tr = matrice_gram(etat, X_tr, X_tr)
    svm = SVC(kernel="precomputed", C=1.0).fit(K_tr, y_tr)
    K_te = matrice_gram(etat, X_te, X_tr)
    lignes.append({"modèle": nom, "famille": "kernel quantique", "paramètres": len(svm.support_),
                   "précision train": svm.score(K_tr, y_tr), "précision test": svm.score(K_te, y_te),
                   "circuits (entraînement)": M_tr * (M_tr - 1) // 2,
                   "circuits (prédiction test)": M_te * len(svm.support_), "optimisation": "convexe"})
    scores_grille[nom] = svm.decision_function(matrice_gram(etat, grille, X_tr))

# --- VQC (entraînés en partie 4) ---
for (arch, L) in [("A — encodage unique", 5), ("B — re-uploading", 5)]:
    r = resultats_vqc[(arch, L)]
    P = r["theta"].size
    nom = f"VQC {arch[0]} (L = {L})"
    lignes.append({"modèle": nom, "famille": "VQC", "paramètres": P + 1,
                   "précision train": precision(r["modele"], r["theta"], r["b"], X_tr_q, y_tr),
                   "précision test": precision(r["modele"], r["theta"], r["b"], X_te_q, y_te),
                   "circuits (entraînement)": len(r["hist"]["loss"]) * M_tr * (2 * P + 1),
                   "circuits (prédiction test)": M_te, "optimisation": "non convexe"})
    scores_grille[nom] = r["modele"](grille_q, r["theta"]) + r["b"]

df_comparaison = pd.DataFrame(lignes)
df_comparaison.style.format({"précision train": "{:.1%}", "précision test": "{:.1%}",
                             "circuits (entraînement)": "{:,}", "circuits (prédiction test)": "{:,}"})

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8.2))
for ax, (nom, s) in zip(axes.ravel(), scores_grille.items()):
    acc = df_comparaison.set_index("modèle").loc[nom, "précision test"]
    tracer_frontiere(ax, s, f"{nom}\nprécision test = {acc:.0%}")
ax_leg = axes.ravel()[-1]
ax_leg.axis("off")
ax_leg.text(0.05, 0.5, "●  points d'entraînement\n▲  points de test\n\ncouleur : score normalisé\n(rouge = classe −1, bleu = +1)\n\ntrait noir : frontière de décision",
            fontsize=11, va="center", transform=ax_leg.transAxes)
plt.tight_layout()
plt.show()

**Ce qu'on observe sur ce découpage.**

**VQC A et kernel « angle » : même classe de fonctions, même résultat.** Les deux modèles partagent l'encodage $S(x)$, et leurs frontières se ressemblent beaucoup. Ce n'est pas un hasard : l'espace de fonctions du noyau angle ($\prod_i \cos^2\frac{x_i - x_i'}{2}$) est exactement celui des polynômes trigonométriques de degré 1 en chaque feature, c'est-à-dire la classe du VQC A identifiée en partie 2. Le kernel y trouve l'optimum régularisé par une optimisation convexe ; le VQC A y arrive par descente de gradient non convexe. Précision test identique, précision train très proche.

**Le kernel IQP est en retrait sur ce découpage.** Sa frontière est organisée autour des coins haut-gauche et bas-droit, là où le produit $x_1 x_2$ est le plus négatif. C'est la signature du terme intriquant $e^{-i\, x_1 x_2\, Z_0 Z_1/2}$ de l'encodage IQP : le noyau « pense » en termes de motifs hyperboliques en $x_1x_2$, une structure qui n'a rien à voir avec la géométrie des deux lunes. Un encodage intriquant n'est pas automatiquement meilleur : ce qui compte, c'est l'adéquation entre le **biais inductif** du noyau et la structure des données.

Avant de conclure, deux vérifications s'imposent : (1) le retrait du kernel IQP vient-il d'un mauvais réglage de l'hyperparamètre $C$ du SVM ? (2) les écarts observés sont-ils significatifs, alors que le test ne compte que 60 points (un point = 1,7 %) ?

In [ ]:
# (1) Réglage de C par validation croisée à 5 plis, sur le seul jeu d'entraînement

grille_C = {"C": [0.1, 1, 10, 100, 1000]}
lignes = []
for nom, etat in [("Kernel quantique angle", etat_angle), ("Kernel quantique IQP", etat_iqp)]:
    K_tr, K_te = matrice_gram(etat, X_tr, X_tr), matrice_gram(etat, X_te, X_tr)
    gs = GridSearchCV(SVC(kernel="precomputed"), grille_C, cv=5).fit(K_tr, y_tr)
    lignes.append({"modèle": nom, "meilleur C": gs.best_params_["C"], "précision CV": gs.best_score_,
                   "précision test": gs.score(K_te, y_te)})
gs = GridSearchCV(SVC(kernel="rbf", gamma="scale"), grille_C, cv=5).fit(X_tr, y_tr)
lignes.append({"modèle": "SVM RBF", "meilleur C": gs.best_params_["C"], "précision CV": gs.best_score_,
               "précision test": gs.score(X_te, y_te)})
pd.DataFrame(lignes).style.format({"précision CV": "{:.1%}", "précision test": "{:.1%}"})

Sur ce découpage, régler $C$ ne rapproche aucun des deux noyaux quantiques du SVM RBF en test. Mais un détail doit alerter : le kernel IQP obtient une précision de validation croisée bien plus haute que sa précision de test. Avec 140 points d'entraînement et 60 de test, **une seule mesure est très bruitée**, et les écarts de quelques points entre modèles ne sont pas interprétables en l'état.

### 6.3 Robustesse : 5 découpages train/test

On refait toute la comparaison sur 5 découpages aléatoires différents (et 5 initialisations différentes pour les VQC). Pour être équitable, le $C$ des trois SVM est réglé par validation croisée à chaque découpage. Les hyperparamètres des VQC (pas d'apprentissage, profondeur, nombre d'itérations) restent fixés, ce qui ne peut que les désavantager. On regarde la moyenne et la dispersion de la précision de test.

In [ ]:
def evaluer_decoupage(graine):
    Xtr_b, Xte_b, ytr_s, yte_s = train_test_split(X_all, y_all, test_size=0.3, random_state=graine, stratify=y_all)
    sc = MinMaxScaler(feature_range=(-onp.pi / 2, onp.pi / 2)).fit(Xtr_b)
    Xtr_s, Xte_s = sc.transform(Xtr_b), sc.transform(Xte_b)
    res = {}
    res["Régression logistique"] = LogisticRegression().fit(Xtr_s, ytr_s).score(Xte_s, yte_s)
    res["MLP (16, 16)"] = (MLPClassifier(hidden_layer_sizes=(16, 16), max_iter=3000, random_state=SEED)
                           .fit(Xtr_s, ytr_s).score(Xte_s, yte_s))
    # SVM : C réglé par validation croisée sur le jeu d'entraînement du découpage
    res["SVM RBF"] = GridSearchCV(SVC(kernel="rbf", gamma="scale"), grille_C, cv=5).fit(Xtr_s, ytr_s).score(Xte_s, yte_s)
    for nom, etat in [("Kernel quantique angle", etat_angle), ("Kernel quantique IQP", etat_iqp)]:
        gs = GridSearchCV(SVC(kernel="precomputed"), grille_C, cv=5).fit(matrice_gram(etat, Xtr_s, Xtr_s), ytr_s)
        res[nom] = gs.score(matrice_gram(etat, Xte_s, Xtr_s), yte_s)
    Xtr_q, Xte_q = pnp.array(Xtr_s, requires_grad=False), pnp.array(Xte_s, requires_grad=False)
    for arch, modele in architectures.items():
        rng = onp.random.default_rng(graine)
        theta = pnp.array(rng.uniform(0, 2 * onp.pi, (5, n_qubits, 3)), requires_grad=True)
        b = pnp.array(0.0, requires_grad=True)
        opt = qml.AdamOptimizer(0.05)
        cout = lambda theta, b: pnp.mean((modele(Xtr_q, theta) + b - ytr_s) ** 2)
        for _ in range(150):
            theta, b = opt.step(cout, theta, b)
        res[f"VQC {arch[0]} (L = 5)"] = precision(modele, theta, b, Xte_q, yte_s)
    return res

t0 = time.time()
scores_decoupages = pd.DataFrame([evaluer_decoupage(g) for g in range(5)])
print(f"({time.time() - t0:.0f} s)")

resume = pd.DataFrame({"moyenne": scores_decoupages.mean(), "écart-type": scores_decoupages.std(),
                       "min": scores_decoupages.min(), "max": scores_decoupages.max()}).sort_values("moyenne")
couleurs = {"classique": "C0", "kernel quantique": "C1", "VQC": "C2"}
famille = df_comparaison.set_index("modèle")["famille"]

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(resume.index, resume["moyenne"], xerr=resume["écart-type"], capsize=4,
        color=[couleurs[famille[m]] for m in resume.index])
for i, (m, ligne) in enumerate(resume.iterrows()):
    ax.scatter(scores_decoupages[m], [i] * len(scores_decoupages), c="k", s=12, zorder=5)
ax.set_xlim(0.7, 1.01)
ax.set_xlabel("précision test (5 découpages : moyenne ± écart-type, points = découpages)")
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c) for c in couleurs.values()], labels=list(couleurs), loc="lower right")
plt.tight_layout()
plt.show()
resume.sort_values("moyenne", ascending=False).style.format("{:.1%}")

**Ce qu'on observe.** Deux groupes se détachent nettement, bien au-delà de la dispersion entre découpages :

1. **~99 % : VQC B (re-uploading), MLP, SVM RBF.** Les modèles assez flexibles pour suivre l'imbrication des lunes. Le VQC à re-uploading fait jeu égal avec les meilleurs modèles classiques.
2. **~86–88 % : régression logistique, kernel angle, kernel IQP, VQC A.** Aucun de ces trois modèles quantiques ne fait mieux qu'un **modèle linéaire**.
   - Pour le VQC A et le kernel angle, c'est la classe de fonctions qui limite : sur des features dans $[-\pi/2, \pi/2]$, un polynôme trigonométrique de degré 1 reste une fonction très lisse, presque linéaire.
   - Le kernel IQP, une fois $C$ réglé, rejoint ce groupe : son intrication ne lui apporte rien sur ce problème, faute d'un biais inductif adapté. C'est aussi le modèle le plus variable d'un découpage à l'autre.

**Leçon de méthode.** Sur le découpage initial, le VQC A et le kernel angle affichaient 91,7 %, et le kernel IQP semblait nettement le plus faible. Ces deux impressions étaient des artefacts d'un seul découpage de 60 points de test. Comparer des modèles de QML (comme n'importe quels modèles) demande plusieurs découpages, plusieurs initialisations et un réglage équitable des hyperparamètres.

### 6.4 Le coût : là où les deux approches quantiques divergent vraiment

À précision comparable, il reste la question du coût sur un vrai processeur (colonnes « circuits » du tableau principal) :
- le **kernel** demande $M(M-1)/2$ circuits pour la matrice de Gram, soit environ $10^4$ ici, puis un circuit par paire (point de test, vecteur support) en prédiction ;
- le **VQC** demande $\text{itérations} \times M \times (2P+1)$ circuits, soit environ $10^6$ ici, mais **un seul circuit par prédiction**.

Avec 140 exemples, le kernel est cent fois moins cher à entraîner. Mais son coût croît en $M^2$, celui du VQC en $M$ : sur un million d'exemples, la matrice de Gram demanderait $5 \times 10^{11}$ circuits (chacun répété des milliers de fois pour les shots). Le choix entre les deux est d'abord une question d'échelle, puis de facilité d'optimisation (convexe d'un côté, non convexe et exposé aux *barren plateaus* de l'autre).

**Et l'avantage quantique ?** Aucun modèle quantique ne bat les meilleurs modèles classiques ici, et c'est attendu : sur des données classiques de faible dimension, il n'y a aucune raison théorique d'en espérer un. Les séparations prouvées entre apprentissage quantique et classique reposent sur des problèmes construits sur mesure, par exemple liés au logarithme discret (Liu, Arunachalam & Temme, 2021). Le message de cette partie n'est donc pas « quel modèle gagne » mais **pourquoi** chaque modèle se comporte comme il le fait : classe de fonctions, biais inductif, coût et optimisation.

---
## 7. Synthèse

| Notion | Ce qu'on a observé | Conséquence pratique |
|---|---|---|
| Modèle linéaire | $f_\theta(x) = \operatorname{Tr}[\rho(x) M_\theta]$ ; l'ansatz fait tourner l'observable sans changer son spectre | Un VQC est un modèle linéaire dans l'espace des états : l'encodage fait tout le travail non linéaire |
| Spectre de Fourier | Les fréquences accessibles sont fixées par l'encodage ; l'erreur résiduelle = énergie des fréquences manquantes | Concevoir l'encodage (répétitions, échelle des features) en fonction de la bande passante voulue |
| Re-uploading | Encodage unique : plafond de loss identique pour L = 3 et 5 ; re-uploading : gain continu | Ajouter des couches sans ré-encoder ne sert à rien au-delà d'un certain point |
| Cône de causalité | À L = 1, le modèle ignorait complètement x₂ | Vérifier que chaque feature peut atteindre l'observable mesurée |
| Parameter-shift | Exacte, sans biais, robuste au bruit de shots, mais 2P + 1 circuits par gradient | Les différences finies sont à proscrire sur matériel ; le coût en circuits domine |
| Barren plateaus | Gradients en 2^(-n) pour les circuits profonds et les coûts globaux | Préférer coûts locaux, circuits peu profonds, initialisation maîtrisée — en gardant en tête le risque de simulabilité classique |
| VQC vs kernel | Même encodage ⇒ même classe de fonctions et même précision (VQC A ≈ kernel angle) ; kernel convexe en O(M²) circuits, VQC non convexe en O(M·P) | Kernel pour les petits datasets, VQC quand M devient grand |
| Biais inductif | Le kernel IQP, pourtant intriquant, ne fait pas mieux qu'une régression logistique : sa structure en x₁x₂ ne colle pas aux données | Un encodage se choisit pour la structure des données, pas pour sa « quanticité » |
| Méthodologie | Un seul découpage surestimait VQC A et kernel angle de ~5 points et sous-estimait le kernel IQP | Plusieurs découpages, plusieurs initialisations, hyperparamètres réglés équitablement |
| Avantage quantique | Aucun sur des données classiques en 2D ; le meilleur VQC égale SVM RBF et MLP | Les avantages prouvés concernent des problèmes construits sur mesure |

### Pour aller plus loin — exercices

1. **Encodage à échelle entraînable.** Remplacer $R_X(x)$ par $R_X(\lambda x)$ avec $\lambda$ entraînable dans la partie 2. Comment le spectre évolue-t-il ? Peut-on ajuster la cible avec moins de répétitions ?
2. **Fonction de perte.** Remplacer la MSE par une entropie croisée sur $p(y{=}1|x) = (1 + f_\theta(x))/2$ et comparer les frontières de la partie 4.
3. **Observable et cône de causalité.** Mesurer $Z_1$, $Z_0 Z_1$ ou $\tfrac12(Z_0 + Z_1)$ au lieu de $Z_0$ dans l'architecture A avec $L = 1$. Lesquelles « voient » les deux features ?
4. **Bruit matériel.** Refaire l'expérience 5.4 sur `default.mixed` en ajoutant un `qml.DepolarizingChannel(p)` après chaque couche. Observer les *noise-induced barren plateaus* : les gradients décroissent maintenant avec la **profondeur**, même à $n$ fixé.
5. **Concentration des kernels.** Pour l'encodage IQP sur $n = 2, \dots, 10$ qubits avec des données aléatoires, tracer la variance des valeurs hors diagonale de la matrice de Gram en fonction de $n$. Comparer à la partie 5.
6. **Ton propre kernel.** Remplacer le kernel IQP de la partie 6 par celui de ton notebook précédent et refaire le face-à-face.

### Références

> Liste établie de mémoire, sans accès à une base bibliographique : vérifie les détails (années, revues, numéros) avant de les citer.

- M. Schuld, R. Sweke, J. J. Meyer — *Effect of data encoding on the expressive power of variational quantum machine learning models*, Phys. Rev. A 103, 032430 (2021).
- A. Pérez-Salinas, A. Cervera-Lierta, E. Gil-Fuster, J. I. Latorre — *Data re-uploading for a universal quantum classifier*, Quantum 4, 226 (2020).
- K. Mitarai, M. Negoro, M. Kitagawa, K. Fujii — *Quantum circuit learning*, Phys. Rev. A 98, 032309 (2018).
- M. Schuld, V. Bergholm, C. Gogolin, J. Izaac, N. Killoran — *Evaluating analytic gradients on quantum hardware*, Phys. Rev. A 99, 032331 (2019).
- J. R. McClean, S. Boixo, V. N. Smelyanskiy, R. Babbush, H. Neven — *Barren plateaus in quantum neural network training landscapes*, Nat. Commun. 9, 4812 (2018).
- M. Cerezo, A. Sone, T. Volkoff, L. Cincio, P. J. Coles — *Cost function dependent barren plateaus in shallow parametrized quantum circuits*, Nat. Commun. 12, 1791 (2021).
- Z. Holmes, K. Sharma, M. Cerezo, P. J. Coles — *Connecting ansatz expressibility to gradient magnitudes and barren plateaus*, PRX Quantum 3, 010313 (2022).
- S. Wang et al. — *Noise-induced barren plateaus in variational quantum algorithms*, Nat. Commun. 12, 6961 (2021).
- E. Grant, L. Wossnig, M. Ostaszewski, M. Benedetti — *An initialization strategy for addressing barren plateaus in parametrized quantum circuits*, Quantum 3, 214 (2019).
- M. Cerezo et al. — *Does provable absence of barren plateaus imply classical simulability?*, arXiv:2312.09121 (2023).
- M. Schuld — *Supervised quantum machine learning models are kernel methods*, arXiv:2101.11020 (2021).
- V. Havlíček et al. — *Supervised learning with quantum-enhanced feature spaces*, Nature 567, 209 (2019).
- S. Thanasilp, S. Wang, M. Cerezo, Z. Holmes — *Exponential concentration in quantum kernel methods*, Nat. Commun. 15, 5200 (2024).
- Y. Liu, S. Arunachalam, K. Temme — *A rigorous and robust quantum speed-up in supervised machine learning*, Nat. Phys. 17, 1013 (2021).